In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import cupy as cp
import numpy as np
from scipy.stats import norm
import pprint
from cupyx.scipy.special import erf

# A set of fixed input
a_value = 1.0
n_psa = 1000
seed = 25

sample_size_now = 397
sample_size_new = 619

wtp = 150000

max_cycle = 200
# Define the cycle range
cycle_range = cp.arange(0, max_cycle + 1)  # 0 to 200 inclusive

def get_discount_factor(cycle_range, dr=0.03, cycles_per_year=16):
    """
    Calculates the discount factor for each cycle based on the discount rate.

    Args:
        cycle_range (array-like): The range of cycles (e.g., np.arange(0, 201)).
        dr (float): The yearly discount rate (default is 0.03, or 3%).
        cycles_per_year (int): The number of cycles in a year (default is 16).

    Returns:
        np.ndarray: An array of discount factors corresponding to the cycle range.
    """
    # Convert yearly discount rate to per-cycle discount rate
    discount_rate_cycle = (1 + dr) ** (1 / cycles_per_year) - 1

    # Calculate the discount factor for each cycle
    discount_factor = 1 / (1 + discount_rate_cycle) ** cycle_range

    return discount_factor

discount_factor = get_discount_factor(cycle_range)

# Hazard ratio for Combo versus Chemo
hr_params = {
    "pfs": {"hr": 0.49, "low": 0.38, "high": 0.63},
    "os": {"hr": 0.60, "low": 0.45, "high": 0.79}
}

# Define the utility dictionary
utility = {
    "stable": 0.75,  # Utility for stable state
    "prog": 0.59     # Utility for progression state
}

# Define the cost dictionary
cost = {
    # Monitoring costs
    "monitoring_stable": 464.85 * 3,
    "monitoring_prog": 1075.49 * 3,

    # Parameters for drug costs
    "surface": 1.86,
    "admin_first": 158.7,
    "admin_sub": 33.6,
    "terminal": 16441.83,
    "price_premetrxed": 7.49,
    "price_cisplatin": 0.18,
    "price_carboplatin": 0.05,
    "dose_premetrxed": 500,
    "dose_cisplatin": 75,
    "dose_carboplatin": 550,
    "dose_drug": 200,

    # Calculated costs
    "intro_cost": (
        1.86 * 500 * 7.49 +
        0.277 * 1.86 * 75 * 0.18 +
        (1 - 0.277) * 550 * 0.05
    ),
    "maintain_cost_chemo": 1.86 * 500 * 7.49
}

In [ ]:
value_based_price_loc = "/content/drive/MyDrive/Colab Notebooks/01_Research/01_Local_Confirmatory_RCT/03_output/02_value_price"
value_based_price = pd.read_csv(f"{value_based_price_loc}/value_price.csv").iloc[0,0]

surv_params_path = "/content/drive/MyDrive/Colab Notebooks/01_Research/01_Local_Confirmatory_RCT/03_output/01_surv_params"
surv_params = pd.read_csv(f"{surv_params_path}/surv_params.csv")
case_params = {
    row["case_name"]: {
        "intercept": row["intercept"],
        "log_scale": row["log_scale"]
    }
    for _, row in surv_params.iterrows()
}

In [ ]:
import cupyx.scipy.special as cpx_special

def norm_ppf(p):
    """
    Compute the percent point function (inverse CDF) for a standard normal distribution.

    Args:
        p (float or cp.ndarray): Probability value(s).

    Returns:
        cp.ndarray: The quantile corresponding to p.
    """
    return cp.sqrt(2) * cpx_special.erfinv(2 * p - 1)

def hr_log_transform(hr: float,
                     hr_lower: float,
                     hr_upper: float,
                     a_value: float = a_value) -> dict:
    """
    Compute the mean and standard deviation of the log-transformed hazard ratio (HR)
    using GPU-accelerated operations via CuPy.

    Parameters:
        hr (float): Hazard ratio estimate.
        hr_lower (float): Lower bound of the confidence interval.
        hr_upper (float): Upper bound of the confidence interval.
        a_value (float): Adjustment value (default from global scope).

    Returns:
        dict: Contains the mean of log(HR) and two versions of its standard deviation.
    """
    # Convert the inputs to GPU arrays (this is optional for scalars, but clarifies intent)
    hr_gpu = cp.asarray(hr)
    hr_lower_gpu = cp.asarray(hr_lower)
    hr_upper_gpu = cp.asarray(hr_upper)

    # Compute the log-transformed values on the GPU
    log_hr = cp.log(hr_gpu)
    log_ci_lower = cp.log(hr_lower_gpu)
    log_ci_upper = cp.log(hr_upper_gpu)

    # Approximate the standard deviation using the confidence interval width,
    # with GPU-accelerated arithmetic.
    sigma_log_hr_old = (log_ci_upper - log_ci_lower) / (2 * norm_ppf(1 - 0.05 / 2))
    sigma_log_hr_updated = cp.sqrt(sigma_log_hr_old**2 / a_value)

    return {
        "mu_log_hr": log_hr,
        "sigma_log_hr": sigma_log_hr_old,
        "sigma_log_hr_updated": sigma_log_hr_updated
    }

In [ ]:
def get_simparams(
    a_value: float = a_value,
    sample_size_now: int = sample_size_now,
    sample_size_new: int = sample_size_new,
    hr_params: dict = hr_params,
    n_psa: int = n_psa,
    seed_for_prior: int = seed
) -> dict:
    """
    Generate prior and posterior hazard ratio (HR) samples for PFS and OS,
    leveraging GPU-accelerated operations with CuPy.

    Returns:
        dict: A nested dictionary with keys ["prior"]["pfs"], ["prior"]["os"],
              ["posterior"]["pfs"], ["posterior"]["os"]. For the posterior,
              each outcome key maps to a dictionary whose keys index the PSA replicate,
              each containing an array of HR samples (all arrays are CuPy arrays).
    """

    # Set the random seed for reproducibility (global state)
    cp.random.seed(seed_for_prior)

    # Initialize the results dictionary.
    res = {
        "prior": {},
        "posterior": {}
    }

    # Compute log-HR parameters (mu and sigma) for PFS and OS using the GPU-accelerated function.
    log_hr_dict = {
        "pfs": hr_log_transform(
            hr_params["pfs"]["hr"],
            hr_params["pfs"]["low"],
            hr_params["pfs"]["high"],
            a_value=a_value
        ),
        "os": hr_log_transform(
            hr_params["os"]["hr"],
            hr_params["os"]["low"],
            hr_params["os"]["high"],
            a_value=a_value
        )
    }

    # Process each outcome (PFS and OS)
    for outcome, params in log_hr_dict.items():
        # These are GPU scalars (0-d CuPy arrays)
        prior_mean       = params["mu_log_hr"]
        prior_sd         = params["sigma_log_hr"]
        prior_sd_updated = params["sigma_log_hr_updated"]

        # Compute the precision (inverse variance) of the original prior distribution.
        prior_precision = 1.0 / (prior_sd ** 2)

        # Draw PSA samples from the updated prior distribution.
        # The result is a GPU array of shape (n_psa,)
        prior_mean_updated_samples = cp.random.normal(
            loc=prior_mean,
            scale=prior_sd_updated,
            size=n_psa
        )

        # Exponentiate to obtain hazard ratio samples for the prior.
        prior_hr_arr = cp.exp(prior_mean_updated_samples)

        # Population variance of the log(HR) scaled by sample_size_now.
        pop_var = (prior_sd ** 2) * sample_size_now
        # The sample variance for the sample mean in new data.
        sample_var = pop_var / sample_size_new
        sample_precision = 1.0 / sample_var

        # For each PSA replicate, simulate new data.
        # Generate a (n_psa x sample_size_new) array:
        new_data_samples = cp.random.normal(
            loc=prior_mean_updated_samples.reshape(n_psa, 1),
            scale=cp.sqrt(sample_var),
            size=(n_psa, sample_size_new)
        )
        # Compute the sample mean for each replicate (along axis 1).
        X_bar = new_data_samples.mean(axis=1)

        # Normal-Normal Bayesian update for each replicate.
        post_mean = (prior_precision * prior_mean + sample_precision * X_bar) / (prior_precision + sample_precision)
        post_sd = prior_sd * cp.sqrt(sample_var / (sample_var + prior_sd ** 2))

        # Draw posterior log(HR) samples for each replicate.
        # Generate an (n_psa x n_psa) array: each row i contains n_psa samples from N(post_mean[i], post_sd).
        post_samples = cp.random.normal(
            loc=post_mean.reshape(n_psa, 1),
            scale=post_sd,
            size=(n_psa, n_psa)
        )
        # Exponentiate to obtain HR samples.
        post_hr_matrix = cp.exp(post_samples)

        # Build a dictionary for posterior samples so that each PSA replicate's results
        # can be accessed individually.
        post_hr_dict = {i: post_hr_matrix[i, :] for i in range(n_psa)}

        # Save the prior and posterior results for this outcome.
        res["prior"][outcome] = prior_hr_arr
        res["posterior"][outcome] = post_hr_dict

    return res

In [ ]:
pprint.pprint(get_simparams(n_psa = 5))

{'posterior': {'os': {0: array([0.51212417, 0.55119926, 0.55881978, 0.5247639 , 0.60169997]),
                      1: array([0.47725361, 0.56083788, 0.45568026, 0.54334585, 0.57465575]),
                      2: array([0.66166305, 0.60054329, 0.69389962, 0.64627995, 0.60853013]),
                      3: array([0.76711865, 0.66096263, 0.61682642, 0.57889588, 0.55666563]),
                      4: array([0.5087286 , 0.56391432, 0.52028336, 0.53485565, 0.54350345])},
               'pfs': {0: array([0.37629594, 0.49134485, 0.42722483, 0.44681428, 0.41146485]),
                       1: array([0.46791085, 0.48208302, 0.46034719, 0.42271456, 0.39882929]),
                       2: array([0.52579593, 0.50958759, 0.45429478, 0.56477073, 0.49243882]),
                       3: array([0.46469704, 0.46648409, 0.46853144, 0.4634213 , 0.52796698]),
                       4: array([0.50237835, 0.49814423, 0.51991546, 0.50124633, 0.5410557 ])}},
 'prior': {'os': array([0.54759304, 0.46491979, 0.62

In [ ]:
def get_nmb(
    hr_pair: dict,
    case_params: dict,
    price_drug: float,
    wtp: float,
    utility: dict,
    cost: dict,
    cycle_range: np.ndarray,
    discount_factor: np.ndarray
) -> tuple:
    """
    Calculate net monetary benefit (NMB) and related economic outcomes
    for combination therapy vs. standard chemotherapy using GPU acceleration.

    Parameters
    ----------
    hr_pair : dict
        Hazard ratios for PFS and OS, e.g., {"pfs": float, "os": float}.
    case_params : dict
        Survival parameters for the chemo arms (used for both arms with HR adjustments),
        e.g.:
            {
                "pfs_chemo": {"intercept": ..., "log_scale": ...},
                "os_chemo":  {"intercept": ..., "log_scale": ...}
            }
    price_drug : float
        Price of the new drug per dose.
    wtp : float
        Willingness-to-pay threshold (per QALY).
    utility : dict
        Utilities for "stable" and "prog" states, e.g., {"stable": 0.75, "prog": 0.59}.
    cost : dict
        Cost parameters.
    cycle_range : np.ndarray
        Range of cycles for the simulation (e.g., np.arange(0, max_cycle+1)).
    discount_factor : np.ndarray
        Discount factors corresponding to each cycle.

    Returns
    -------
    tuple
        (Avg_cost_per_month_chemo, Avg_cost_per_month_combo,
         MB_chemo, MB_combo, NMB, ICER)
    """

    # Define a GPU version of the normal CDF using the error function.
    def norm_cdf(x):
        return 0.5 * (1.0 + erf(x / cp.sqrt(2.0)))

    # Nested survival simulation function.
    def simulate_survival(cycle_range, intercept, log_scale, hr=None):
        """
        Compute survival probabilities per cycle.
        Uses a log-normal approximation:
          surv_ctrl = 1 - Φ( (log(time) - intercept) / exp(log_scale) )
        If hr is provided, the control curve is modified via exponentiation.
        """
        # Adjust cycle: each cycle is ~3/4 of a month.
        adjusted_cycle = cycle_range * (3.0 / 4.0)
        # Compute the baseline (chemo) survival curve.
        surv_ctrl = 1 - norm_cdf((cp.log(adjusted_cycle + 1e-10) - intercept) / cp.exp(log_scale))
        # Force the last cycle's survival probability to 0.
        surv_ctrl[-1] = 0
        # If a hazard ratio is provided, modify the survival curve.
        if hr is not None:
            surv_prob = surv_ctrl ** hr
        else:
            surv_prob = surv_ctrl
        surv_prob[-1] = 0
        return surv_prob

    # --- 2. Simulate survival curves for chemotherapy (chemo) ---
    pfs_chemo = simulate_survival(
        cycle_range,
        intercept=case_params["pfs_chemo"]["intercept"],
        log_scale=case_params["pfs_chemo"]["log_scale"]
    )
    os_chemo = simulate_survival(
        cycle_range,
        intercept=case_params["os_chemo"]["intercept"],
        log_scale=case_params["os_chemo"]["log_scale"]
    )

    # --- 3. Simulate survival curves for combination therapy (combo) ---
    # (Using the chemo parameters and applying the hazard ratios.)
    pfs_combo = simulate_survival(
        cycle_range,
        intercept=case_params["pfs_chemo"]["intercept"],
        log_scale=case_params["pfs_chemo"]["log_scale"],
        hr=hr_pair["pfs"]
    )
    os_combo = simulate_survival(
        cycle_range,
        intercept=case_params["os_chemo"]["intercept"],
        log_scale=case_params["os_chemo"]["log_scale"],
        hr=hr_pair["os"]
    )

    # --- 4. Define state probabilities for chemotherapy ---
    prog_chemo = cp.maximum(os_chemo - pfs_chemo, 0)
    stable_chemo = pfs_chemo
    dead_chemo = 1 - os_chemo

    # Force final cycle values.
    stable_chemo[-1] = 0
    prog_chemo[-1] = 0
    dead_chemo[-1] = 1

    # --- 5. Calculate QALYs for chemotherapy ---
    # Each cycle is ~3/4 of a month; convert to QALYs per cycle.
    qalys_chemo = cp.sum(
        stable_chemo * (utility["stable"] * 3 / (4 * 12)) * discount_factor +
        prog_chemo   * (utility["prog"]   * 3 / (4 * 12)) * discount_factor
    )

    # --- 6. Compute costs for chemotherapy ---
    monitoring_chemo = cost["monitoring_stable"] * stable_chemo + cost["monitoring_prog"] * prog_chemo
    admin_chemo = cp.zeros(len(cycle_range))
    admin_chemo[:37] = stable_chemo[:37] * (cost["admin_first"] + cost["admin_sub"])
    final_chemo = cp.concatenate((dead_chemo[:1], cp.diff(dead_chemo))) * cost["terminal"]
    treatment_chemo = cp.zeros(len(cycle_range))
    treatment_chemo[:4] = cost["intro_cost"] * stable_chemo[:4]
    treatment_chemo[4:36] = cost["maintain_cost_chemo"] * stable_chemo[4:36]

    costs_chemo = cp.sum((monitoring_chemo + admin_chemo + final_chemo + treatment_chemo) * discount_factor)

    # --- 7. Define state probabilities for combination therapy ---
    prog_combo = cp.maximum(os_combo - pfs_combo, 0)
    stable_combo = pfs_combo
    dead_combo = 1 - os_combo

    stable_combo[-1] = 0
    prog_combo[-1] = 0
    dead_combo[-1] = 1

    # --- 8. Calculate QALYs for combination therapy ---
    qalys_combo = cp.sum(
        stable_combo * (utility["stable"] * 3 / (4 * 12)) * discount_factor +
        prog_combo   * (utility["prog"]   * 3 / (4 * 12)) * discount_factor
    )

    # --- 9. Compute costs for combination therapy ---
    drug_cost_intro    = cost["dose_drug"] * price_drug + cost["intro_cost"]
    drug_cost_maintain = cost["dose_drug"] * price_drug + cost["maintain_cost_chemo"]

    monitoring_combo = cost["monitoring_stable"] * stable_combo + cost["monitoring_prog"] * prog_combo
    admin_combo = cp.zeros(len(cycle_range))
    admin_combo[:37] = stable_combo[:37] * (cost["admin_first"] + cost["admin_sub"])
    final_combo = cp.concatenate((dead_combo[:1], cp.diff(dead_combo))) * cost["terminal"]
    treatment_combo = cp.zeros(len(cycle_range))
    treatment_combo[:4] = drug_cost_intro * stable_combo[:4]
    treatment_combo[4:36] = drug_cost_maintain * stable_combo[4:36]

    costs_combo = cp.sum((monitoring_combo + admin_combo + final_combo + treatment_combo) * discount_factor)

    # --- 10. Derive cost-effectiveness outcomes ---
    # Average cost per month (each cycle ~3 weeks; scale by 4/3 to monthly)
    Avg_cost_per_month_chemo = (costs_chemo / len(cycle_range)) * (4 / 3)
    Avg_cost_per_month_combo  = (costs_combo / len(cycle_range)) * (4 / 3)

    # Monetary benefit per arm
    MB_chemo = wtp * qalys_chemo - costs_chemo
    MB_combo = wtp * qalys_combo - costs_combo

    # Net Monetary Benefit (NMB) and Incremental Cost-Effectiveness Ratio (ICER)
    NMB = MB_combo - MB_chemo
    delta_cost = costs_combo - costs_chemo
    delta_qaly = qalys_combo - qalys_chemo
    if delta_qaly.item() == 0:
        ICER = cp.inf if delta_cost.item() > 0 else 0
    else:
        ICER = delta_cost / delta_qaly

    return (
        Avg_cost_per_month_chemo,
        Avg_cost_per_month_combo,
        MB_chemo,
        MB_combo,
        NMB,
        ICER
    )

In [ ]:
def compute_psa_nmb(
    a_value: float = a_value,
    sample_size_now: int = sample_size_now,
    sample_size_new: int = sample_size_new,
    n_psa: int = n_psa,
    seed_for_prior: int = seed,
    price_drug: float = value_based_price,
    wtp: float = wtp,
    case_params: dict = case_params,
    hr_params: dict = hr_params,
    utility: dict = utility,
    cost: dict = cost,
    cycle_range: np.ndarray = cycle_range,
    discount_factor: np.ndarray = discount_factor
) -> dict:
    """
    Simulates cost and net monetary benefit (NMB) metrics for prior and posterior
    hazard-ratio draws, returning each result as a 1D NumPy array of length n_psa.

    For the posterior, the get_simparams function returns a dictionary of
    n_psa arrays (each subarray is length n_psa). We compute the average
    across each subarray, so each posterior result ends up as an n_psa array.
    """

    # 1) Get hazard-ratio draws from get_simparams (GPU-accelerated)
    psa_params = get_simparams(
        a_value=a_value,
        sample_size_now=sample_size_now,
        sample_size_new=sample_size_new,
        hr_params=hr_params,
        n_psa=n_psa,
        seed_for_prior=seed_for_prior
    )
    # Expected structure:
    # {
    #   "prior": {
    #       "pfs": cp.ndarray(n_psa),
    #       "os":  cp.ndarray(n_psa)
    #   },
    #   "posterior": {
    #       "pfs": {0: cp.ndarray(n_psa), 1: cp.ndarray(n_psa), ...},
    #       "os":  {0: cp.ndarray(n_psa), 1: cp.ndarray(n_psa), ...}
    #   }
    # }

    # 2) Prepare arrays for prior results (on CPU)
    Avg_cost_per_month_chemo_prior = np.zeros(n_psa)
    Avg_cost_per_month_combo_prior  = np.zeros(n_psa)
    MB_chemo_prior                  = np.zeros(n_psa)
    MB_combo_prior                  = np.zeros(n_psa)

    # 3) Prepare arrays for posterior results (each as length n_psa)
    Avg_cost_per_month_chemo_post = np.zeros(n_psa)
    Avg_cost_per_month_combo_post = np.zeros(n_psa)
    MB_chemo_post                 = np.zeros(n_psa)
    MB_combo_post                 = np.zeros(n_psa)

    # -------------------------------------------------------------------------
    # 4) Compute Prior-based Results: one hazard ratio draw per iteration.
    # -------------------------------------------------------------------------
    for i in range(n_psa):
        # Extract hazard ratios (GPU scalars) and convert them to Python floats.
        hr_pair_prior = {
            "pfs": float(psa_params["prior"]["pfs"][i].item()),
            "os":  float(psa_params["prior"]["os"][i].item())
        }
        # Call the GPU-accelerated get_nmb.
        (chemo_cost,
         combo_cost,
         mb_chemo,
         mb_combo,
         _nmb,
         _icer) = get_nmb(
            hr_pair=hr_pair_prior,
            case_params=case_params,
            price_drug=price_drug,
            wtp=wtp,
            utility=utility,
            cost=cost,
            cycle_range=cycle_range,
            discount_factor=discount_factor
        )
        # Convert GPU scalar outputs to CPU floats.
        Avg_cost_per_month_chemo_prior[i] = float(chemo_cost.item())
        Avg_cost_per_month_combo_prior[i]  = float(combo_cost.item())
        MB_chemo_prior[i]                  = float(mb_chemo.item())
        MB_combo_prior[i]                  = float(mb_combo.item())

    # -------------------------------------------------------------------------
    # 5) Compute Posterior-based Results: for each i, we have a sub-array of length n_psa.
    #    We compute the average across each sub-array.
    # -------------------------------------------------------------------------
    for i in range(n_psa):
        # Each of these is a GPU array of shape (n_psa,)
        post_pfs_arr = psa_params["posterior"]["pfs"][i]
        post_os_arr  = psa_params["posterior"]["os"][i]

        # Prepare subarrays (on CPU) to store each mini-PSA iteration’s results.
        chemo_cost_sub = np.zeros(n_psa)
        combo_cost_sub = np.zeros(n_psa)
        MB_chemo_sub   = np.zeros(n_psa)
        MB_combo_sub   = np.zeros(n_psa)

        # For each mini-PSA replicate.
        for j in range(n_psa):
            hr_pair_post = {
                "pfs": float(post_pfs_arr[j].item()),
                "os":  float(post_os_arr[j].item())
            }
            (chemo_cost,
             combo_cost,
             mb_chemo,
             mb_combo,
             _nmb,
             _icer) = get_nmb(
                hr_pair=hr_pair_post,
                case_params=case_params,
                price_drug=price_drug,
                wtp=wtp,
                utility=utility,
                cost=cost,
                cycle_range=cycle_range,
                discount_factor=discount_factor
            )
            chemo_cost_sub[j] = float(chemo_cost.item())
            combo_cost_sub[j] = float(combo_cost.item())
            MB_chemo_sub[j]   = float(mb_chemo.item())
            MB_combo_sub[j]   = float(mb_combo.item())

        # Average across the n_psa mini-PSA draws for this posterior replicate.
        Avg_cost_per_month_chemo_post[i] = np.mean(chemo_cost_sub)
        Avg_cost_per_month_combo_post[i] = np.mean(combo_cost_sub)
        MB_chemo_post[i]                 = np.mean(MB_chemo_sub)
        MB_combo_post[i]                 = np.mean(MB_combo_sub)

    # -------------------------------------------------------------------------
    # 6) Return the final structure (all arrays are 1D NumPy arrays of length n_psa)
    # -------------------------------------------------------------------------
    return {
        "Avg_cost_per_month_chemo_prior": Avg_cost_per_month_chemo_prior,
        "Avg_cost_per_month_combo_prior": Avg_cost_per_month_combo_prior,
        "MB_chemo_prior": MB_chemo_prior,
        "MB_combo_prior": MB_combo_prior,

        "Avg_cost_per_month_chemo_post": Avg_cost_per_month_chemo_post,
        "Avg_cost_per_month_combo_post": Avg_cost_per_month_combo_post,
        "MB_chemo_post": MB_chemo_post,
        "MB_combo_post": MB_combo_post
    }

In [ ]:
def get_ev(
    a_value: float = a_value,
    sample_size_now: int = sample_size_now,
    sample_size_new: int = sample_size_new,
    n_psa: int = n_psa,
    seed_for_prior: int = seed,
    price_drug: float = value_based_price,
    wtp: float = wtp,
    case_params: dict = case_params,
    hr_params: dict = hr_params,
    utility: dict = utility,
    cost: dict = cost,
    cycle_range: np.ndarray = cycle_range,
    discount_factor: np.ndarray = discount_factor
) -> dict:
    """
    Calculate the expected value of sampling (EV) metrics, including per-person
    and population-level Expected Value of Perfect Information (EVPI) and Expected
    Value of Sample Information (EVSI), as well as the expected net benefit of sampling (ENBS).

    The function:
      1. Obtains PSA (probabilistic sensitivity analysis) results using GPU‐accelerated
         simulation (via compute_psa_nmb).
      2. Extracts the monetary benefits (MB) from both prior and posterior analyses.
      3. Computes per-person EVPI and EVSI.
      4. Computes the trial’s expected net benefit (ENBS) and then scales EVPI/EVSI to the
         population level.

    Returns:
        dict: A dictionary with keys:
              "EVPI_per_person", "EVSI_per_person", "EVPI", "EVSI", "ENBS"
    """

    # 0) Fixed parameters
    incidence = 71794         # Annual incidence
    prevalence = 640488       # Prevalence
    cost_per_sample = 48324.48  # Cost per sample taken in the trial
    uptake_rate = 0.4         # Proportion of incident cases receiving the intervention
    d_r = 0.03                # Annual discount rate

    # 1) Get PSA results using the GPU-accelerated simulation
    psa_results = compute_psa_nmb(
        a_value=a_value,
        sample_size_now=sample_size_now,
        sample_size_new=sample_size_new,
        n_psa=n_psa,
        seed_for_prior=seed_for_prior,
        price_drug=price_drug,
        wtp=wtp,
        case_params=case_params,
        hr_params=hr_params,
        utility=utility,
        cost=cost,
        cycle_range=cycle_range,
        discount_factor=discount_factor
    )

    # 2) Extract relevant monetary benefit arrays from PSA results
    MB_chemo_prior = psa_results["MB_chemo_prior"]
    MB_combo_prior = psa_results["MB_combo_prior"]
    MB_chemo_post  = psa_results["MB_chemo_post"]
    MB_combo_post  = psa_results["MB_combo_post"]

    # 3) Calculate per-person EVPI
    # For each PSA replicate, choose the maximum MB (i.e. best strategy) from the prior analysis.
    max_MB_prior = np.maximum(MB_chemo_prior, MB_combo_prior)
    EVPI_1 = np.mean(max_MB_prior)
    EVPI_2 = max(np.mean(MB_chemo_prior), np.mean(MB_combo_prior))
    EVPI_per_person = EVPI_1 - EVPI_2

    # 4) Calculate per-person EVSI
    max_MB_post = np.maximum(MB_chemo_post, MB_combo_post)
    EVSI_1 = np.mean(max_MB_post)
    EVSI_2 = EVPI_2
    EVSI_per_person = EVSI_1 - EVSI_2

    # 5) Calculate the Expected Net Benefit of Sampling (ENBS)
    MB_chemo_post_mean = np.mean(MB_chemo_post)
    MB_combo_post_mean = np.mean(MB_combo_post)

    # Net benefits during the trial period (assumed to be a 3-year trial)
    # - Part 1: Benefit for the treated sample (half receive each treatment)
    part1 = (sample_size_new / 2.0) * (MB_chemo_post_mean + MB_combo_post_mean)
    # - Part 2: Benefit for prevalent patients not included in the trial and incident patients during the trial
    part2 = MB_chemo_post_mean * ((prevalence - sample_size_new) +
              np.sum(incidence * (1 / (1 + d_r) ** np.arange(0, 3, 1))))
    # - Part 3: Benefit after the trial (from year 3 to 10) with different uptake rates
    part3 = (
        EVSI_1 * np.sum(incidence * uptake_rate * (1 / (1 + d_r) ** np.arange(3, 10, 1))) +
        MB_chemo_post_mean * np.sum(incidence * (1 - uptake_rate) * (1 / (1 + d_r) ** np.arange(3, 10, 1)))
    )
    # - Part 4: Trial cost
    part4 = cost_per_sample * sample_size_new
    # - Part 5: Opportunity cost for not conducting the trial
    part5 = MB_chemo_post_mean * (prevalence + np.sum(incidence * (1 / (1 + d_r) ** np.arange(0, 10, 1))))

    # ENBS expressed in millions
    ENBS = (part1 + part2 + part3 - part4 - part5) / 1_000_000

    # 6) (Optional) Enforce conditions: If any of the per-person EV metrics are non-positive,
    #    you could set them to NaN. (This block is currently commented out.)
    # if (
    #     np.isnan(EVPI_per_person) or np.isnan(EVSI_per_person) or
    #     (EVPI_per_person <= 0) or (EVSI_per_person <= 0) or
    #     (EVPI_per_person <= EVSI_per_person)
    # ):
    #     EVPI_per_person = np.nan
    #     EVSI_per_person = np.nan

    # 7) Calculate population-level EVPI and EVSI over a 10-year horizon
    discount_factors = 1 / (1 + d_r) ** np.arange(0, 10, 1)
    EVPI = EVPI_per_person * np.sum(incidence * discount_factors) / 1_000_000
    EVSI = EVSI_per_person * np.sum(incidence * discount_factors) / 1_000_000

    # 8) Return the results in a dictionary
    return {
        "EVPI_per_person": EVPI_per_person,
        "EVSI_per_person": EVSI_per_person,
        "EVPI": EVPI,
        "EVSI": EVSI,
        "ENBS": ENBS
    }

In [ ]:
get_ev(a_value=0.5)

In [ ]:
get_ev(a_value=0.75)

In [ ]:
get_ev(a_value=1.0)

In [ ]:
# a value from 0.5 to 1.0 by 0.1
a_value_arr = np.arange(0.5, 1.1, 0.1)

# price range
price_drug_arr = np.sort(np.append(np.arange(10, 20.5, 0.5), value_based_price))

# sample_size_new range
sample_size_new_arr = np.sort(np.append(np.arange(300, 1100, 100), sample_size_new))

In [ ]:
!pip install tqdm joblib tqdm_joblib
import itertools
from joblib import Parallel, delayed
from tqdm.notebook import tqdm
from tqdm_joblib import tqdm_joblib

def run_single_combination(a_val, drug_price, ss_new):
    out = get_ev(
        a_value=a_val,
        price_drug=drug_price,
        sample_size_new=ss_new
    )
    return {
        "a_value": a_val,
        "price_drug": drug_price,
        "sample_size_new": ss_new,
        "EVPI_pp": out["EVPI_per_person"],
        "EVSI_pp": out["EVSI_per_person"],
        "EVPI": out["EVPI"],
        "EVSI": out["EVSI"],
        "ENBS": out["ENBS"]
    }

def run_grid_ev_parallel(
    a_value_arr,
    price_drug_arr,
    sample_size_new_arr
) -> pd.DataFrame:
    """
    Evaluates all combinations of (a_value, price_drug, sample_size_new) in parallel,
    displaying a progress bar, and returns a DataFrame of EVPI/EVSI results.
    """

    # 1) Generate the Cartesian product of the three arrays
    all_combinations = list(
        itertools.product(a_value_arr, price_drug_arr, sample_size_new_arr)
    )
    total_combos = len(all_combinations)

    # 2) Use tqdm_joblib to show a progress bar as we run the parallel jobs
    with tqdm_joblib(tqdm(total=total_combos, desc="Computing EVPI & EVSI & ENBS")):
        results = Parallel(n_jobs=-1)(
            delayed(run_single_combination)(a_val, drug_price, ss_new)
            for (a_val, drug_price, ss_new) in all_combinations
        )

    # 3) Convert list of dicts to a DataFrame
    df = pd.DataFrame(results)
    return df

/usr/local/lib/python3.11/dist-packages/tqdm_joblib/__init__.py:4: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [ ]:
# Run the parallel analysis
df_evpi_evsi = run_grid_ev_parallel(a_value_arr, price_drug_arr, sample_size_new_arr)

# Inspect the results
df_evpi_evsi.head()

# export path
enbs_paths = "/content/drive/MyDrive/Colab Notebooks/01_Research/01_Local_Confirmatory_RCT/03_output/03_enbs"
# export it to a CSV file
df_evpi_evsi.to_csv(f"{enbs_paths}/value_price.csv", index=False)

Computing EVPI & EVSI & ENBS:   0%|          | 0/1386 [00:00<?, ?it/s]

  0%|          | 0/1386 [00:00<?, ?it/s]